# Mouse heart P8, H3K27ac ChIP-seq, MI vs sham, day 3 (GSE123867 / PRJNA510092)
Result 1 (AP-1 spacing). Kaggle notebook: **Internet ON, CPU**.

Steps: download FASTQ from ENA (MD5-checked) -> fastp -> Bowtie 2 -> MAPQ >= 30, duplicates removed -> MACS3 broad
peaks -> injury-responsive enhancers (RREs) and control pool (`01_call_injury_responsive_enhancers.py`, control
signal >= median) -> primary AP-1 spacing test (`spacing_core.py` = step 02).
Output: `enhancers_results.tgz` (rre.bed, control_pool.bed, summary.json, spacing.json, QC, peaks).

d0 = sham, d1 = MI; in_* = input. FASTQ and index are placed in /tmp because /kaggle/working (20 GB) is too small for `samtools sort`.

In [ ]:
%%bash
set -e
apt-get -qq update > /dev/null && apt-get -qq install -y bowtie2 samtools bedtools unzip > /dev/null
pip -q install macs3
wget -q http://opengene.org/fastp/fastp -O /usr/local/bin/fastp && chmod +x /usr/local/bin/fastp
for t in twoBitToFa faToTwoBit; do
  wget -q https://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64/$t -O /usr/local/bin/$t && chmod +x /usr/local/bin/$t
done
bowtie2 --version | head -1; macs3 --version; nproc; free -g | head -2

In [ ]:
%%bash
mkdir -p /tmp/fq /tmp/idx && ln -sfn /tmp/fq fq && ln -sfn /tmp/idx idx && df -h /tmp /kaggle/working

In [ ]:
%%writefile 01_call_injury_responsive_enhancers.py
"""Result 1, step 01: injury-responsive enhancers (RREs) and the control pool from H3K27ac ChIP-seq.

Input : MACS3 broadPeak files per sample, filtered BAM per sample (uninjured R1 R2, injured R1 R2), TSS BED.
Consensus peaks = merged peaks of all samples; signal = CPM per kb per replicate.
RRE    : log2FC(injured/uninjured) >= 1 in BOTH replicate pairs (R1-R1, R2-R2), injured signal >= 75th percentile,
         > 2 kb from the nearest TSS.
Control: |mean log2FC| < 0.3 and signal >= the --ctl-quantile quantile at both time points, > 2 kb from TSS.
         The paper uses --ctl-quantile 0.5 (median) for every ChIP-seq set.
Output : rre.bed, control_pool.bed (chrom start end name lfc inj tss_dist), consensus.bed, summary.json.

Run inside the Kaggle notebooks in notebooks/ (zebrafish heart, killifish fin, mouse P1/P8) and on the
zebrafish fin BAM files (see README). Requires bedtools and samtools on PATH.
"""
import argparse
import io
import json
import os
import subprocess

import numpy as np
import pandas as pd


def sh(cmd):
    return subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True).stdout


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--peaks", nargs="+", required=True)
    ap.add_argument("--bam0", nargs=2, required=True, help="uninjured BAM, replicate 1 and 2")
    ap.add_argument("--bam1", nargs=2, required=True, help="injured BAM, replicate 1 and 2")
    ap.add_argument("--tss", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--ctl-quantile", dest="ctl_q", type=float, default=0.75,
                    help="control signal threshold (the paper uses 0.5)")
    ap.add_argument("--chrom-regex", dest="chrom_re", default=r"^chr[0-9]+\b",
                    help="chromosomes used (UCSC: ^chr[0-9]+; NCBI: ^NC_)")
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    cons = os.path.join(a.out, "consensus.bed")
    sh(f"cat {' '.join(a.peaks)} | cut -f1-3 | grep -E '{a.chrom_re}' | sort -k1,1 -k2,2n | bedtools merge -i - > {cons}")
    bams = a.bam0 + a.bam1
    cov = sh(f"bedtools multicov -bams {' '.join(bams)} -bed {cons}")
    df = pd.read_csv(io.StringIO(cov), sep="\t", header=None,
                     names=["chrom", "start", "end", "c0a", "c0b", "c1a", "c1b"])
    lib = {c: float(sh(f"samtools view -c {b}").strip()) for c, b in zip(["c0a", "c0b", "c1a", "c1b"], bams)}
    for c in lib:
        df[c] = df[c] / lib[c] * 1e6 / ((df.end - df.start) / 1000)  # CPM per kb
    ps = 0.1
    df["lfc_a"] = np.log2((df.c1a + ps) / (df.c0a + ps))
    df["lfc_b"] = np.log2((df.c1b + ps) / (df.c0b + ps))
    df["lfc"] = (df.lfc_a + df.lfc_b) / 2
    df["inj"] = (df.c1a + df.c1b) / 2
    df["uni"] = (df.c0a + df.c0b) / 2
    p75_1, p75_0 = df.inj.quantile(0.75), df.uni.quantile(0.75)
    q1, q0 = df.inj.quantile(a.ctl_q), df.uni.quantile(a.ctl_q)

    tss = pd.read_csv(a.tss, sep="\t", header=None, usecols=[0, 1], names=["chrom", "pos"])
    df["tss_dist"] = np.inf
    for c, g in df.groupby("chrom"):
        t = np.sort(tss.loc[tss.chrom == c, "pos"].values)
        if len(t) < 2:
            continue
        mid = ((g.start + g.end) // 2).values
        j = np.searchsorted(t, mid).clip(1, len(t) - 1)
        df.loc[g.index, "tss_dist"] = np.minimum(np.abs(mid - t[j - 1]), np.abs(mid - t[j]))
    distal = df.tss_dist > 2000
    rre = df[(df.lfc_a >= 1) & (df.lfc_b >= 1) & (df.inj >= p75_1) & distal].copy()
    ctl = df[(df.lfc.abs() < 0.3) & (df.inj >= q1) & (df.uni >= q0) & distal].copy()
    rre["name"] = [f"RRE_{i}" for i in range(len(rre))]
    ctl["name"] = [f"CTL_{i}" for i in range(len(ctl))]
    cols = ["chrom", "start", "end", "name", "lfc", "inj", "tss_dist"]
    rre[cols].to_csv(os.path.join(a.out, "rre.bed"), sep="\t", header=False, index=False)
    ctl[cols].to_csv(os.path.join(a.out, "control_pool.bed"), sep="\t", header=False, index=False)
    s = {"consensus_peaks": len(df), "rre": len(rre), "control_pool": len(ctl), "lib_sizes": lib,
         "p75_inj": float(p75_1), "p75_uni": float(p75_0), "ctl_quantile": a.ctl_q,
         "rep_concordance_spearman": float(df[["lfc_a", "lfc_b"]].corr(method="spearman").iloc[0, 1])}
    json.dump(s, open(os.path.join(a.out, "summary.json"), "w"), indent=2)
    print(json.dumps(s, indent=2))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile spacing_core.py
"""AP-1 spacing test for one enhancer set (the primary test of Result 1).

Units: 1 kb centred on each enhancer. Canonical AP-1 sites: regex TGA[CG]TCA (overlapping matches allowed).
Every injury-responsive enhancer (RRE) with >= 2 AP-1 sites is matched, without replacement, to a control
enhancer with exactly the same number of AP-1 sites and the closest GC content (RREs visited in random order,
seed 7). Statistic: fraction of AP-1-AP-1 pairs (start-to-start distance) spaced 7-60 bp, RRE minus control.
Null: permutation of group labels at the enhancer level (5,000 permutations, one-sided). Secondary: KS test,
sub-ranges 7-30 and 30-60 bp.

Usage (step 02): python spacing_core.py --rre RRE.bed --ctrl CONTROL_POOL.bed --genome-2bit GENOME.2bit
                 --genome-sizes GENOME.sizes --out spacing.json
Input BED columns (from 01_call_injury_responsive_enhancers.py): chrom start end name lfc inj tss_dist.
Output: {"ap1_pair_spacing_test": {...}}.
Requires UCSC twoBitToFa on PATH.
"""
import argparse
import json
import os
import re
import subprocess

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp

AP1 = re.compile(r"(?=(TGA[CG]TCA))")


def fetch_ap1_stream(bed, twobit, tmp):
    """Memory-light: stream the FASTA and keep only AP-1 positions and GC content per unit."""
    bed[["chrom", "start", "end", "name"]].to_csv(tmp + ".bed", sep="\t", header=False, index=False)
    subprocess.run(f"twoBitToFa -bed={tmp}.bed {twobit} {tmp}.fa", shell=True, check=True)
    out, name, buf = {}, None, []

    def flush():
        if name is not None:
            sq = "".join(buf)
            n_acgt = sum(sq.count(b) for b in "ACGT")
            out[name] = (ap1(sq), (sq.count("G") + sq.count("C")) / max(1, n_acgt))

    with open(tmp + ".fa") as fh:
        for l in fh:
            l = l.strip()
            if l.startswith(">"):
                flush()
                name, buf = l[1:].split(":")[0], []
            else:
                buf.append(l.upper())
    flush()
    os.remove(tmp + ".fa")
    return out


def ap1(s):
    return [m.start() for m in AP1.finditer(s)]


def pairs(pos):
    return [b - a for i, a in enumerate(pos) for b in pos[i + 1:]]


def frac_close(ds, lo=7, hi=60):
    ds = np.asarray(ds)
    return float(((ds >= lo) & (ds <= hi)).mean()) if len(ds) else np.nan


def l2_test(rre_pos, ctl_pos, nperm, seed):
    """rre_pos/ctl_pos: list of lists of AP-1 positions per unit. Permutation of labels at the unit level."""
    units = rre_pos + ctl_pos
    lab = np.array([1] * len(rre_pos) + [0] * len(ctl_pos), bool)
    up = [pairs(p) for p in units]

    def stat(l):
        a = [d for u, x in zip(up, l) if x for d in u]
        b = [d for u, x in zip(up, l) if not x for d in u]
        return frac_close(a) - frac_close(b), a, b

    obs, a, b = stat(lab)
    rng = np.random.default_rng(seed)
    cnt = sum(stat(rng.permutation(lab))[0] >= obs for _ in range(nperm))
    return {"frac_7_60_rre": frac_close(a), "frac_7_60_ctl": frac_close(b), "diff": obs,
            "sens_7_30": [frac_close(a, 7, 30), frac_close(b, 7, 30)],
            "sens_30_60": [frac_close(a, 30, 60), frac_close(b, 30, 60)],
            "perm_p_one_sided": (cnt + 1) / (nperm + 1), "ks_p": float(ks_2samp(a, b).pvalue),
            "n_units": [len(rre_pos), len(ctl_pos)], "n_pairs": [len(a), len(b)]}


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--rre", required=True)
    ap.add_argument("--ctrl", required=True)
    ap.add_argument("--genome-2bit", dest="genome_2bit", required=True)
    ap.add_argument("--genome-sizes", dest="genome_sizes", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--nperm", type=int, default=5000)
    ap.add_argument("--seed", type=int, default=7)
    a = ap.parse_args()
    os.makedirs(os.path.dirname(a.out) or ".", exist_ok=True)
    tmp = a.out + ".tmp"
    res = {}

    # controls matched on AP-1 count from the whole control pool
    cols = ["chrom", "start", "end", "name", "lfc", "inj", "tss_dist"]
    rre = pd.read_csv(a.rre, sep="\t", header=None, names=cols)
    pool = pd.read_csv(a.ctrl, sep="\t", header=None, names=cols)
    for t_ in (rre, pool):
        mid = (t_.start + t_.end) // 2
        t_["start"] = (mid - 500).clip(lower=0)
        t_["end"] = t_.start + 1000
    zsz = pd.read_csv(a.genome_sizes, sep="\t", header=None, names=["chrom", "len"]).set_index("chrom").len
    rre = rre[rre.end <= rre.chrom.map(zsz)].reset_index(drop=True)
    pool = pool[pool.end <= pool.chrom.map(zsz)].reset_index(drop=True)
    fr = fetch_ap1_stream(rre, a.genome_2bit, tmp + "_zr")
    fp = fetch_ap1_stream(pool, a.genome_2bit, tmp + "_zp")
    rre["pos"], rre["gc"] = rre.name.map(lambda n: fr[n][0]), rre.name.map(lambda n: fr[n][1])
    pool["pos"], pool["gc"] = pool.name.map(lambda n: fp[n][0]), pool.name.map(lambda n: fp[n][1])
    del fr, fp
    rre["k"], pool["k"] = rre.pos.map(len), pool.pos.map(len)
    R = rre[rre.k >= 2].reset_index(drop=True)
    P = pool[pool.k >= 2].reset_index(drop=True)
    used = np.zeros(len(P), bool)
    ctl_pos, rre_pos, unmatched = [], [], 0
    for r in R.sample(frac=1, random_state=a.seed).itertuples():
        cand = np.where((P.k.values == r.k) & ~used)[0]
        if len(cand) == 0:
            unmatched += 1
            continue
        j = cand[np.argmin(np.abs(P.gc.values[cand] - r.gc))]
        used[j] = True
        rre_pos.append(r.pos)
        ctl_pos.append(P.pos[j])
    l2 = l2_test(rre_pos, ctl_pos, a.nperm, a.seed)
    l2.update({"rre_units_ge2": int(len(R)), "pool_units_ge2": int(len(P)), "unmatched": unmatched,
               "matched_fraction": (len(R) - unmatched) / max(len(R), 1)})
    res["ap1_pair_spacing_test"] = l2
    json.dump(res, open(a.out, "w"), indent=2, default=float)
    print(json.dumps(res, indent=2, default=float))


if __name__ == "__main__":
    main()


## Reference: mm10 Bowtie 2 index (iGenomes) and UCSC files

In [ ]:
%%bash
set -e
mkdir -p idx ref
IG=https://ngi-igenomes.s3.amazonaws.com/igenomes/Mus_musculus/UCSC/mm10/Sequence/Bowtie2Index
for f in genome.1.bt2 genome.2.bt2 genome.3.bt2 genome.4.bt2 genome.rev.1.bt2 genome.rev.2.bt2; do wget -q $IG/$f -O idx/$f; done
U=https://hgdownload.soe.ucsc.edu/goldenPath/mm10
wget -q $U/bigZips/mm10.2bit -O ref/genome.2bit
wget -q $U/bigZips/mm10.chrom.sizes -O ref/genome.sizes
wget -q $U/database/refGene.txt.gz -O ref/refGene.txt.gz
zcat ref/refGene.txt.gz | awk -F'\t' 'BEGIN{OFS="\t"}{p=($4=="+")?$5:$6; print $3,p,p+1,$13,".",$4}' | sort -u > ref/tss.bed
wc -l ref/tss.bed; bowtie2-inspect -n idx/genome | head -2

## FASTQ from ENA

In [ ]:
import subprocess, hashlib, concurrent.futures as cf, urllib.request, csv, io, os
os.makedirs('fq', exist_ok=True)
want = ['SRR8318839', 'SRR8318840', 'SRR8318843', 'SRR8318844', 'SRR8318845', 'SRR8318846', 'SRR8318849', 'SRR8318850']
api = ('https://www.ebi.ac.uk/ena/portal/api/filereport?accession=PRJNA510092&result=read_run'
       '&fields=run_accession,fastq_ftp,fastq_md5&format=tsv')
rows = [r for r in csv.DictReader(io.StringIO(urllib.request.urlopen(api).read().decode()), delimiter='\t')
        if r['run_accession'] in want]
assert len(rows) == len(want), (len(rows), len(want))
def get(r):
    f = 'fq/' + r['run_accession'] + '.fastq.gz'
    for _ in range(5):
        subprocess.run(['wget', '-q', '-c', 'https://' + r['fastq_ftp'].split(';')[0], '-O', f])
        m = hashlib.md5()
        with open(f, 'rb') as fh:
            for chunk in iter(lambda: fh.read(1 << 24), b''):
                m.update(chunk)
        if m.hexdigest() == r['fastq_md5'].split(';')[0]:
            return r['run_accession'], 'OK'
        os.remove(f)
    return r['run_accession'], 'MD5_FAILED'
with cf.ThreadPoolExecutor(3) as ex:
    for res in ex.map(get, rows):
        print(*res)

## Read processing

In [ ]:
%%bash
set -e
mkdir -p bam qc
proc() {  # $1 = sample name, remaining arguments = runs
  n=$1; shift; [ -s bam/$n.bam ] && return
  cat $(for r in "$@"; do echo fq/$r.fastq.gz; done) > fq/$n.cat.fq.gz
  fastp -w 4 -i fq/$n.cat.fq.gz -o fq/$n.trim.fq.gz -j qc/$n.fastp.json -h qc/$n.fastp.html 2> qc/$n.fastp.log
  bowtie2 -p 4 -x idx/genome -U fq/$n.trim.fq.gz 2> qc/$n.bowtie2.log \
    | samtools view -@2 -b -q 30 -F 4 - | samtools sort -@2 -o bam/$n.sorted.bam -
  samtools markdup -@2 -r -s bam/$n.sorted.bam bam/$n.bam 2> qc/$n.markdup.log
  samtools index bam/$n.bam; rm -f bam/$n.sorted.bam fq/$n.cat.fq.gz fq/$n.trim.fq.gz
  echo "$n: $(samtools view -c bam/$n.bam) reads; $(tail -1 qc/$n.bowtie2.log)"
}
proc d0_R1 SRR8318845
proc d0_R2 SRR8318846
proc d1_R1 SRR8318839
proc d1_R2 SRR8318840
proc in_d0_R1 SRR8318849
proc in_d0_R2 SRR8318850
proc in_d1_R1 SRR8318843
proc in_d1_R2 SRR8318844

## MACS3 broad peaks against input

In [ ]:
%%bash
set -e
mkdir -p peaks
macs3 callpeak -t bam/d0_R1.bam -c bam/in_d0_R1.bam -f BAM -g mm --broad -n d0_R1 --outdir peaks 2> peaks/d0_R1.log
macs3 callpeak -t bam/d0_R2.bam -c bam/in_d0_R2.bam -f BAM -g mm --broad -n d0_R2 --outdir peaks 2> peaks/d0_R2.log
macs3 callpeak -t bam/d1_R1.bam -c bam/in_d1_R1.bam -f BAM -g mm --broad -n d1_R1 --outdir peaks 2> peaks/d1_R1.log
macs3 callpeak -t bam/d1_R2.bam -c bam/in_d1_R2.bam -f BAM -g mm --broad -n d1_R2 --outdir peaks 2> peaks/d1_R2.log
wc -l peaks/*.broadPeak

## Enhancers and AP-1 spacing test

In [ ]:
%%bash
set -e
python 01_call_injury_responsive_enhancers.py --peaks peaks/d0_R1_peaks.broadPeak peaks/d0_R2_peaks.broadPeak \
  peaks/d1_R1_peaks.broadPeak peaks/d1_R2_peaks.broadPeak \
  --bam0 bam/d0_R1.bam bam/d0_R2.bam --bam1 bam/d1_R1.bam bam/d1_R2.bam --tss ref/tss.bed \
  --ctl-quantile 0.5 --chrom-regex '^chr([0-9]+|X)\b' --out enhancers
python spacing_core.py --rre enhancers/rre.bed --ctrl enhancers/control_pool.bed \
  --genome-2bit ref/genome.2bit --genome-sizes ref/genome.sizes --out enhancers/spacing.json > /dev/null
python -c "
import json; s=json.load(open('enhancers/summary.json')); f=json.load(open('enhancers/spacing.json'))['ap1_pair_spacing_test']
print('RREs', s['rre'], 'control pool', s['control_pool'], 'replicate concordance', round(s['rep_concordance_spearman'],3))
print('AP-1 spacing test:', json.dumps(f))"
tar czf enhancers_results.tgz enhancers qc peaks/*.broadPeak peaks/*.log
rm -rf fq idx